In [5]:
import pandas as pd

train = pd.read_csv("../processed/train.csv")
val   = pd.read_csv("../processed/val.csv")
test  = pd.read_csv("../processed/test.csv")

print(train.shape)
print(train.head())

(350, 22)
   segment_length_km  is_monsoon_zone  is_midc_zone  is_bridge  hour_of_day  \
0           0.090142                0             0          0           15   
1           0.124398                0             0          0           15   
2           0.116054                0             0          0           16   
3           0.108687                0             0          0            0   
4           0.120051                0             0          0           12   

   day_of_week  is_peak_hour  base_speed_kmh  congestion_multiplier  \
0            6             0              40               0.973642   
1            4             0              40               0.779911   
2            1             0              40               1.297752   
3            0             0              40               1.292132   
4            5             0              40               0.739220   

   eta_minutes  ...  road_type_primary  road_type_primary_link  \
0     0.500000  ...   

In [6]:
TARGET = "eta_minutes"

# Everything except eta_minutes = input features
feature_cols = [c for c in train.columns if c != TARGET]

X_train = train[feature_cols]
y_train = train[TARGET]

X_val = val[feature_cols]
y_val = val[TARGET]

X_test = test[feature_cols]
y_test = test[TARGET]

print("Features:", X_train.shape)
print("Target sample:", y_train.head())

Features: (350, 21)
Target sample: 0    0.500000
1    1.933482
2    0.973349
3    0.500000
4    0.500000
Name: eta_minutes, dtype: float64


In [7]:
X_train = X_train.replace({"True": 1, "False": 0})
X_val   = X_val.replace({"True": 1, "False": 0})
X_test  = X_test.replace({"True": 1, "False": 0})

In [ ]:
# Convert categorical → numeric
X_train = pd.get_dummies(X_train)
X_val   = pd.get_dummies(X_val)
X_test  = pd.get_dummies(X_test)

# Align columns
X_train, X_val = X_train.align(X_val, join='left', axis=1, fill_value=0)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# Train model
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Validate
val_preds = model.predict(X_val)
mae = mean_absolute_error(y_val, val_preds)
print(f"Validation MAE: {mae:.3f} minutes")

Validation MAE: 0.406 minutes


In [9]:
test_preds = model.predict(X_test)

if y_test is not None:
    from sklearn.metrics import mean_absolute_error
    test_mae = mean_absolute_error(y_test, test_preds)
    print(f"Test MAE: {test_mae:.3f} minutes")
else:
    print("Test predictions generated (no ground truth available).")

Test MAE: 0.427 minutes


In [10]:
print([col for col in X_train.columns if "eta" in col.lower()])

[]


In [11]:
print(pd.DataFrame({
    "Actual": y_test[:10],
    "Predicted": test_preds[:10]
}))

     Actual  Predicted
0  1.009891   0.642054
1  0.500000   1.035252
2  0.500000   1.313996
3  0.500000   0.787192
4  0.737663   0.727990
5  0.500000   0.556616
6  0.500000   1.390179
7  0.681690   1.126781
8  0.744936   0.756053
9  0.500000   0.965341


In [12]:
print(X_train.shape, X_test.shape)

(350, 21) (75, 21)


In [13]:
import joblib
import os

os.makedirs("models/trained", exist_ok=True)
joblib.dump(model, "models/trained/rf_model.pkl")
print("Model saved successfully!") 

Model saved successfully!
